# GeneCompass Pretraining Notebook
This notebook demonstrates pretraining of the GeneCompass model, which incorporates biological prior knowledge into a BERT-based architecture for gene sequence modeling.
## Key Components
1. **Prior Knowledge Integration**: Uses five types of biological knowledge embeddings,
2. **Distributed Training**: Optimized for multi-GPU setups,
3. **Memory Optimization**: Gradient checkpointing and mixed precision,
4. **Dynamic Batching**: Length-based grouping for efficient training


## Environment Setup
First we configure the environment paths and distributed training settings:

In [1]:
import os
import sys

# Set project root - MODIFY THIS PATH TO MATCH YOUR ENVIRONMENT
project_root = "/path/to/GeneCompass-main/"
sys.path.append(project_root)

# Configure distributed training environment
os.environ["NCCL_DEBUG"] = "INFO"
os.environ["OMPI_MCA_opal_cuda_support"] = "true"
os.environ["CONDA_OVERRIDE_GLIBC"] = "2.56"
os.environ["WANDB_MODE"] = "offline"

# Simulate distributed training environment for single-GPU notebook
os.environ["LOCAL_RANK"] = "0"
os.environ["RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12348"

## Import Dependencies
Import required libraries and custom modules:

In [2]:
import pickle
import random
import datetime
import subprocess
import numpy as np
import pytz
import argparse

import torch
import torch.distributed as dist

from transformers import BertConfig, TrainingArguments
from datasets import load_from_disk, disable_caching
disable_caching()

# Import custom GeneCompass modules
from genecompass import GenecompassPretrainer, BertForMaskedLM
from genecompass.utils import load_prior_embedding

## Pretraining Function
Main training logic with distributed setup and model configuration:

In [3]:
def main(args):
    # Set seeds for reproducibility
    random.seed(args.seed_num)
    np.random.seed(args.seed_num)
    torch.manual_seed(args.seed_val)
    torch.cuda.manual_seed_all(args.seed_val)

    # Distributed training setup
    args.local_rank = int(os.environ["LOCAL_RANK"])
    args.world_rank = int(os.environ["RANK"])
    args.world_size = int(os.environ["WORLD_SIZE"])

    # Configure output directories
    training_output_dir = f"{args.output_directory}/models/{args.run_name}/"
    logging_dir = f"{args.output_directory}/runs/{args.run_name}/"
    model_output_dir = os.path.join(training_output_dir, "models/")

    # Create output directories
    if args.world_rank == 0:
        os.makedirs(training_output_dir, exist_ok=True)
        os.makedirs(model_output_dir, exist_ok=True)

    # Load token dictionary
    with open(args.token_dict_path, "rb") as fp:
        token_dictionary = pickle.load(fp)

    # Load biological prior knowledge embeddings
    knowledges = dict()
    out = load_prior_embedding(token_dictionary_or_path=args.token_dict_path)
    knowledges['promoter'] = out[0]
    knowledges['co_exp'] = out[1]
    knowledges['gene_family'] = out[2]
    knowledges['peca_grn'] = out[3]
    knowledges['homologous_gene_human2mouse'] = out[4]

    # Model configuration (BERT-base architecture)
    config = {
        "hidden_size": 768,
        "num_hidden_layers": 12,
        "initializer_range": 0.02,
        "layer_norm_eps": 1e-12,
        "attention_probs_dropout_prob": 0.02,
        "hidden_dropout_prob": 0.02,
        "intermediate_size": 3072,
        "hidden_act": "gelu",
        "max_position_embeddings": 2048,
        "model_type": "bert",
        "num_attention_heads": 12,
        "pad_token_id": token_dictionary.get("<pad>"),
        "vocab_size": len(token_dictionary),
        "use_values": True,
        "use_promoter": True,
        "use_co_exp": True,
        "use_gene_family": True,
        "use_peca_grn": True,
        "warmup_steps": args.warmup_steps,
        "emb_warmup_steps": args.emb_warmup_steps,
        "use_cls_token": True,
    }

    # Initialize model
    model_config = BertConfig(**config)
    model = BertForMaskedLM(model_config, knowledges=knowledges)
    model.train()

    # Training configuration
    training_args = TrainingArguments(
        run_name=args.run_name,
        fp16=args.fp16,
        fp16_opt_level="O1",
        ddp_find_unused_parameters=False,
        gradient_checkpointing=args.gradient_checkpointing,
        dataloader_num_workers=args.dataloader_num_workers,
        learning_rate=args.max_learning_rate,
        do_train=args.do_train,
        do_eval=args.do_eval,
        group_by_length=True,
        length_column_name="length",
        disable_tqdm=False,
        lr_scheduler_type=args.lr_scheduler_type,
        warmup_steps=args.warmup_steps,
        weight_decay=args.weight_decay,
        per_device_train_batch_size=args.train_micro_batch_size_per_gpu,
        num_train_epochs=args.num_train_epochs,
        save_strategy="steps" if args.save_model else None,
        save_steps=args.save_steps if args.save_model else None,
        logging_steps=100,
        output_dir=training_output_dir,
        logging_dir=logging_dir,
    )

    # Load dataset
    train_dataset = load_from_disk(args.dataset_directory)
    example_lengths_file = os.path.join(args.dataset_directory, 'sorted_length.pickle')

    # Initialize trainer
    trainer = GenecompassPretrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        example_lengths_file=example_lengths_file,
        token_dictionary=token_dictionary,
    )

    # Start training
    trainer.train()

    # Save final model
    if args.save_model:
        trainer.save_model(model_output_dir)

## Configuration & Execution
Set parameters and start training. Modify paths according to your environment:


In [4]:
class Args:
    pass

args = Args()

# Required parameters
args.run_name = "test_run"
args.token_dict_path = "../../prior_knowledge/human_mouse_tokens.pickle"
args.dataset_directory = "../../data/cell_type_annotation/hMS/train"
args.output_directory = "../../PretrainingOutputs/"

# Training hyperparameters
args.seed_num = 0
args.seed_val = 42
args.num_train_epochs = 5
args.train_micro_batch_size_per_gpu = 1
args.max_learning_rate = 5e-5
args.warmup_steps = 10000
args.emb_warmup_steps = 10000
args.lr_scheduler_type = "linear"
args.weight_decay = 0.01

# System configuration
args.dataloader_num_workers = 0
args.do_train = True
args.do_eval = False
args.save_model = True
args.save_strategy = "steps"
args.save_steps = 100000
args.fp16 = True
args.gradient_checkpointing = False

# Initialize distributed training
dist.init_process_group(
    backend="nccl",
    init_method="env://",
    world_size=int(os.environ["WORLD_SIZE"]),
    rank=int(os.environ["RANK"]),
)

# Start training
main(args)

Now using GeneCompass model!!!


/opt/miniconda3/envs/GeneCompass/lib/python3.12/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss
100,2.512200
200,2.374800
300,2.331100
400,2.291900
500,2.253200
600,2.213200
700,2.160200
800,2.100200
900,2.048100
1000,2.016800


user:1859973:1859973 [0] NCCL INFO Bootstrap : Using enp59s0f0:10.12.0.18<0>
user:1859973:1859973 [0] NCCL INFO NET/Plugin: No plugin found (libnccl-net.so)
user:1859973:1859973 [0] NCCL INFO NET/Plugin: Plugin load returned 2 : libnccl-net.so: cannot open shared object file: No such file or directory : when loading libnccl-net.so
user:1859973:1859973 [0] NCCL INFO NET/Plugin: Using internal network plugin.
user:1859973:1859973 [0] NCCL INFO cudaDriverVersion 12040
NCCL version 2.21.5+cuda12.4
user:1859973:1859973 [0] NCCL INFO Comm config Blocking set to 1
user:1859973:1860803 [0] NCCL INFO Failed to open libibverbs.so[.1]
user:1859973:1860803 [0] NCCL INFO NET/Socket : Using [0]enp59s0f0:10.12.0.18<0> [1]br-fdce3f129cef:172.18.0.1<0> [2]veth1d56971:fe80::68:15ff:fe45:7da9%veth1d56971<0> [3]vethe2cf6f7:fe80::2884:dbff:fede:2670%vethe2cf6f7<0> [4]vethef47ccd:fe80::7820:aeff:fe50:42f3%vethef47ccd<0> [5]veth9aca980:fe80::58dc:a2ff:fef5:11d4%veth9aca980<0> [6]vethd6ca2b5:fe80::6c01:ddff:f